# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")


✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202405_Flood_Brasil'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'blackmarble_hd'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 18 .tif files in the S3 bucket.


['drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_BRDF.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_Cloud.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A3_Brazil2024_A2024061_March2024.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_BRDF_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_Cloud_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024128_May8_2024_BRDF_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024128_May8_2024_Cloud_Large.tif',
 'drcs_activations/202405_Fl

## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 45
  - Total size: 16.81 GB

📁 Cached files (first 10):
  - drcs_activations/202402_Fire_Guatemala/sentinel2/swir/S2B_shortwaveInfrared_20240223_162159_T15PYS.tif (86.3 MB)
  - drcs_activations/202402_Fire_Guatemala/sentinel2/true/S2B_trueColor_20240223_162159_T15PYS.tif (345.1 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T020715Z_20240206T130219Z_S1A_30_v0.1_B01_WTR.tif (1.6 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T140849Z_20240206T130818Z_S1A_30_v0.1_B01_WTR.tif (0.9 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T020715Z_20240206T022545Z_S1A_30_v0.1_B01_WTR.tif (1.6 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T140849Z_20240206T134347Z_S1A_30_v0.1_B01_WTR.tif (0.9 MB)
  - drcs_activations/202402_Flood_CA/aria_opera/water_change_map_t035_20240206

(45, 18044418239)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys

['drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_BRDF.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_Cloud.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A3_Brazil2024_A2024061_March2024.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_BRDF_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_Cloud_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024128_May8_2024_BRDF_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024128_May8_2024_Cloud_Large.tif',
 'drcs_activations/202405_Fl

In [15]:
# Define filename creator functions for different file types
def create_cog_filename_blackmarble_doy(f, EVENT_NAME):
    """Convert day of year (YYYYDOY) to date format and move to end of filename."""
    from datetime import datetime, timedelta
    import re
    from pathlib import Path
    
    filename = Path(f).stem
    extension = Path(f).suffix
    
    # Find the AYYYYDOY pattern (e.g., A2024127)
    doy_pattern = r'A(\d{4})(\d{3})'
    match = re.search(doy_pattern, filename)
    
    if match:
        year = int(match.group(1))
        doy = int(match.group(2))
        
        # Convert DOY to date
        date = datetime(year, 1, 1) + timedelta(days=doy - 1)
        formatted_date = date.strftime('%Y-%m-%d')
        
        # Extract the parts we want to keep
        # Look for finalBMHD_VNP46A[2/3] pattern
        product_match = re.search(r'(finalBMHD_VNP46A[23])', filename)
        if product_match:
            product_part = product_match.group(1)
        else:
            product_part = 'finalBMHD'
        
        # Look for all suffixes after the date info
        # This will capture combinations like BRDF, Cloud, Large, BRDF_Large, Cloud_Large
        suffixes = []
        
        # Check for BRDF or Cloud
        type_match = re.search(r'_(BRDF|Cloud)', filename)
        if type_match:
            suffixes.append(type_match.group(1))
        
        # Check for Large (separate check to catch both "Large" alone and "BRDF_Large", "Cloud_Large")
        if '_Large' in filename:
            suffixes.append('Large')
        
        # Join suffixes with underscore
        suffix_part = '_'.join(suffixes) if suffixes else ''
        
        # Create new filename
        if suffix_part:
            cog_filename = f'{EVENT_NAME}_{product_part}_{suffix_part}_{formatted_date}_day{extension}'
        else:
            cog_filename = f'{EVENT_NAME}_{product_part}_{formatted_date}_day{extension}'
    else:
        # Fallback
        cog_filename = f'{EVENT_NAME}_{filename}{extension}'
    
    return cog_filename


filter_str = 'blackmarble_hd/finalBMHD'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_blackmarble_doy(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-06_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-06_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-07_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-07_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-08_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-08_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-09_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-09_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-10_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-10_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-11_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-11_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-14_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-14_

In [16]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_blackmarble_doy, 
                                target_dir = "Blackmarble", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-06_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-06_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-07_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-07_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-08_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-08_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-09_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-09_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-10_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-10_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-11_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-11_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-14_day.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-14_da

Reading input: /tmp/tmpmocmx0_8_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmprm8jb1fa.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-06_day.tif
   [MEMORY] Final: 1808.6 MB (Change: +1509.0 MB)


...

✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_Large_2024-05-14_day.tif

[14/15] Processing: drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024135_May15_2024_Cloud_Large.tif
   Output filename: 202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_Large_2024-05-14_day.tif
   [MEMORY] Initial: 1837.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=151321/151321
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Conv

Reading input: /tmp/tmpiwzgrxd4_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpr93d88lj.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202405_Flood_Brasil_finalBMHD_VNP46A3_Large_2024-03-01_day.tif
   [MEMORY] Final: 1840.5 MB (Change: +2.9 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_finalBMHD_VNP46A3_Large_2024-03-01_day.tif

✅ Batch processing complete: 15 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Blackmarble/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Blackmarble/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 15
Successful: 15
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T20:31:58.206631


In [17]:
keys

['drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_BRDF.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_Cloud.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A3_Brazil2024_A2024061_March2024.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_BRDF_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_Cloud_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024128_May8_2024_BRDF_Large.tif',
 'drcs_activations/202405_Flood_Brasil/blackmarble_hd/finalBMHD_VNP46A2_Brazil2024_A2024128_May8_2024_Cloud_Large.tif',
 'drcs_activations/202405_Fl

In [22]:
def rename_with_datetime_suffix(filepath, EVENT_NAME=None):
    """Extract datetime from directory path and append to filename."""
    import re
    from pathlib import Path
    
    path = Path(filepath)
    filename_stem = path.stem
    extension = path.suffix
    
    # Extract datetime from the directory path
    # Looking for pattern like "20240508T185236Z" in "BMHD_Brazil2024-20240508T185236Z-001"
    datetime_pattern = r'(\d{8}T\d{6}Z)'
    match = re.search(datetime_pattern, str(path))
    
    if match:
        datetime_str = match.group(1)
        # Format datetime as YYYY-MM-DD-THH:MM:SSZ
        # Convert 20240508T185236Z to 2024-05-08-T18:52:36Z
        year = datetime_str[0:4]
        month = datetime_str[4:6]
        day = datetime_str[6:8]
        hour = datetime_str[9:11]
        minute = datetime_str[11:13]
        second = datetime_str[13:15]
        formatted_datetime = f'{year}-{month}-{day}-T{hour}:{minute}:{second}Z'
        
        # Clean the filename by removing Brazil2024_A2024127_May7_2024 patterns
        cleaned_filename = filename_stem
        
        # Remove patterns like "Brazil2024_A2024127_May7_2024" or similar
        # This pattern captures: CountryYear_AYYYYDDD_MonthDay_Year
        cleaned_filename = re.sub(r'_Brazil\d{4}_A\d{7}_[A-Za-z]+\d+_\d{4}', '', cleaned_filename)
        
        # Also handle patterns like "_Brazil2024_A2024061_March2024"
        cleaned_filename = re.sub(r'_Brazil\d{4}_A\d{7}_[A-Za-z]+\d{4}', '', cleaned_filename)
        
        # Create new filename with formatted datetime at the end
        if EVENT_NAME:
            new_filename = f'{EVENT_NAME}_{cleaned_filename}_{formatted_datetime}{extension}'
        else:
            new_filename = f'{cleaned_filename}_{formatted_datetime}{extension}'
    else:
        # If no datetime found, still try to clean the filename
        cleaned_filename = filename_stem
        cleaned_filename = re.sub(r'_Brazil\d{4}_A\d{7}_[A-Za-z]+\d+_\d{4}', '', cleaned_filename)
        cleaned_filename = re.sub(r'_Brazil\d{4}_A\d{7}_[A-Za-z]+\d{4}', '', cleaned_filename)
        
        if EVENT_NAME:
            new_filename = f'{EVENT_NAME}_{cleaned_filename}{extension}'
        else:
            new_filename = f'{cleaned_filename}{extension}'
    
    # Return just the filename (not the full path)
    return new_filename

filter_str = 'blackmarble_hd/BMHD_Brazil'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx, i in enumerate(filter_):
    test_wm = rename_with_datetime_suffix(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_2024-05-08-T18:52:36Z.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_2024-05-08-T18:52:36Z.tif
  202405_Flood_Brasil_finalBMHD_VNP46A3_2024-05-08-T18:52:36Z.tif


In [23]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = rename_with_datetime_suffix, 
                                target_dir = "Blackmarble", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_2024-05-08-T18:52:36Z.tif
  202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_2024-05-08-T18:52:36Z.tif
  202405_Flood_Brasil_finalBMHD_VNP46A3_2024-05-08-T18:52:36Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202405_Flood_Brasil/blackmarble_hd
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Blackmarble

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202405_Flood_Brasil

[1/3] Processing: drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_BRDF.tif
   Output filename: 202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_2024-05-08-T18:52:36Z.tif
   [MEMORY] Initial: 1848.6 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIF

Reading input: /tmp/tmp7xesjycu_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp5ty6wief.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_2024-05-08-T18:52:36Z.tif
   [MEMORY] Final: 1848.8 MB (Change: +0.2 MB)


...

✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_finalBMHD_VNP46A2_BRDF_2024-05-08-T18:52:36Z.tif

[2/3] Processing: drcs_activations/202405_Flood_Brasil/blackmarble_hd/BMHD_Brazil2024-20240508T185236Z-001/BMHD_Brazil2024/finalBMHD_VNP46A2_Brazil2024_A2024127_May7_2024_Cloud.tif
   Output filename: 202405_Flood_Brasil_finalBMHD_VNP46A2_Cloud_2024-05-08-T18:52:36Z.tif
   [MEMORY] Initial: 1848.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=1, center sample non-zero=36481/36481
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF 

Reading input: /tmp/tmpyto3hnyz_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp861aw0rp.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Blackmarble/202405_Flood_Brasil_finalBMHD_VNP46A3_2024-05-08-T18:52:36Z.tif
   [MEMORY] Final: 1852.2 MB (Change: +3.4 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202405_Flood_Brasil_finalBMHD_VNP46A3_2024-05-08-T18:52:36Z.tif

✅ Batch processing complete: 3 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Blackmarble/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Blackmarble/files_converted.csv
📁 COGs saved locally to: output/202405_Flood_Brasil

📊 BATCH PROCESSING SUMMARY
Total files processed: 3
Successful: 3
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T20:35:44.731924


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")